# 🧮 Environmental and physiological regulators of cholera
This is a graphical interface for using a somewhat simplified implementation of the model by Koelle et al. (2009) of the effects of seasonality, climate variations and immune period on cholera dynamics.

Koelle *et al*.'s model emerges from a statistical optimization of environmental and physiological parameters, to reflect data from Matlab, Bangladesh.
The key dynamics are:
- **Environmental up- and down-regulation of the transmission rate** (a metric of the likelihood an infected person will pass that infection on to a susceptible person) by short timescale (seasonal) variations, and by longer timescale (climatic) variation in weather.
  
  Rainfall is an important aspect of weather variations, but their analysis also other direct and indirect indicators of exposure.
  The mechanisms underlying observed relationships are complex, in part because they have competing effects.
  For example, increased rainfall potentially increases exposure by flooding sanitation facilities and forcing people to gather in dry areas, but also potentially reduces exposure by flushing *Vibrio cholera* bacteria (the causal agent of cholera) away from population centers. 
- Key physiological characteristics of cholera infections are not known, and are inferred through the statistical analysis:
  - The **asymptomatic ratio**, which is the number of people with undetected cholera infections for each detected case.

    This ratio is important because asymptomatic cases still trigger immunity, and still shed bacteria that can infect others.
  - The **immune period**, which is how long a person who has recovered from cholera retains sufficient immune response to decrease the likelihood of re-infection.

This implementation simplifies the orginal model in several respects, to make it more suitable for modeling exercises:
- The original model uses environmental time series to estimate parameters; this implementation uses their mathemtical formulation but uses hypothetical environmental variations to investigate their impacts on cholera case loads. 
- This implementation makes it possible to recreate dynamics using the parameters found by Koelle *et al*., but also to explore the consequences of different parameter sets.
- Koelle et al. assess multiple co-circulating strains to estimate parameters; this implementation uses the resulting parameters but simulates only one strain (the so-called El Tor biotype).
- Koelle et al. consider partial and temporally varying immunity due to vaccination; this implementation simplifies the effects of vaccination to be the removal of a given fraction of the susceptible pool.
  That is, if the vaccination is only 50% effective at preventing infection, this implementation assumes that the parameter specifying the fraction vaccinated has been discounted by 50%, so that the reduction of the susceptible population is correct.
  It's assumed that the vaccinated fraction is constant through the simulated period.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi, cos, sin, erfc
plot_prefix = "Cholera_plot%d" % (int(10000.*np.random.uniform()))
global nyears_,beta_t_
plt.ion()
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [ ]:
def plot_transmission(nyears = 3,season = 1.,climate=1.5,verbose=False):
    global nyears_,beta_t_
    # We actually calculate in months... 
    nyears_=nyears
    nmonths=12*nyears
    
    # Temporally varying transmission rate
    beta_season=[1.+season*q*(0.1e1-cos(pi*q/0.3e1))/0.8e1 for q in np.arange(1,13)]
    beta_season.reverse()
    #beta_season= [1.+q*(0.1e1-cos(fpi*q/0.3e1))/0.2e1 for q in np.arange(1,13)]
    for i in range(12):
        beta_season[i]=beta_season[i]/beta_season[-1]

    #b0=10.
    climate=max(climate,0.)
    beta_long_term=[q for q in np.linspace(1.5,climate,round(nmonths/2))]
    beta_long_term.extend([q for q in np.linspace(climate,1.5,round(nmonths/2))])
    beta_t=[beta_long_term[i]*beta_season[i % 11] for i in range(nmonths)]
    beta_t_=beta_t

    if verbose:
        print('beta_season = ',beta_season)
        print('beta_long_term = ',beta_long_term)
        print('beta_t = ',beta_t)
        
    # create figure and subpolts
    fig=plt.figure()  #figsize=(20,12))
    # plot seasonal variation in transmission
    ax1=fig.add_subplot(311)
    ax1.plot(np.arange(1,13),beta_season,'r-',label='Seasonal Trans. rate, $\\beta_{season}$')
    #ax1.set_xlabel(r'Months')
    ax1.set_ylabel(r'$\beta_{season}$')
    
    ax2=fig.add_subplot(312)
    ax2.plot(beta_long_term,'b-',label='Climatic Trans. rate,$\\beta_{climate}$')
    #ax2.set_xlabel(r'Months')
    ax2.set_ylabel(r'$\beta_{climate}$')

    ax3=fig.add_subplot(313)
    ax3.plot(range(nmonths),beta_t,'k-',label='Total Trans. rate, $\\beta_t$')
    ax3.set_xlabel(r'Months')
    ax3.set_ylabel(r'$\beta_{t}$')
    #show(trans_plot,gridlines=True)

In [ ]:
nyears=widgets.IntText(value=30,width=10,description = r"$n_{years}$")
season=widgets.FloatText(value=0,description = r"$c$")
climate=widgets.FloatText(value=1.5,description = r"$b$")
ui1 = widgets.VBox([nyears,season,climate])

h = '30px'
L_nyears = widgets.Label(value='Number of years',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="97%",height=h))
L_season = widgets.Label(value='Seasonality factor',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="95%",height=h))
L_climate = widgets.Label(value='Climate factor',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="95%",height=h))
uiL1 = widgets.VBox([L_nyears,L_season,L_climate])

ui1_ = widgets.HBox([uiL1,ui1])

def plotTrans(nyears,season,climate):
    plot_transmission(nyears=nyears,climate=climate,season=season)
    
out = widgets.interactive_output(plotTrans,{'nyears':nyears,'season':season,'climate':climate})
display(ui1_,out)


:::{figure} #kc_1 
:placeholder: images/kc1.png
:align: left
:::

In [ ]:
def plot_caseload(Nint=200000,alpha=.57,gamma_=1.,epsilon = 0.,Iinit_frac=.0001,
                  asymptom=25,t_immunity=12,frac_vacc=0.,
                  show_legend=True,show_immunity=True):
    global nyears_,beta_t_
    nyears=nyears_
    beta_t=beta_t_
    # We actually calculate in months... 
    nmonths=12*nyears
    # We want population as a float
    N=float(Nint)
    
    # Noise
    eps_t=1.+np.random.uniform(low=-epsilon, high=epsilon, size=nmonths)
    #eps_t=1.-epsilon*2.*epsilon*np.random.uniform(low=0.0, high=1.0, size=nmonths)


    #  kappa contains the fractional loss of immunity
    #t_immunity=12 # Period over which immunity declines (in months)
    #kappa=[asymptom*q for q in np.linspace(1.,0.,t_immunity)]
    kappa=[asymptom*0.5*(erfc(q)+0.) for q in np.linspace(-2.,2.,t_immunity)]

    #  Create arrays to hold population categories
    S=[0. for i in range(nmonths)]
    I=[0. for i in range(nmonths+1)]
    R=[0. for i in range(nmonths)]
    years=[q/12. for q in np.arange(nmonths)]

    #  Initial conditions
    I[0]=Iinit_frac*N  #  Initial infecteds
    S[0]=(1.-frac_vacc)*(N-I[0])  #  Initial susceptibles
    R[0]=0.  # Initial resistants

    for t in range(1,nmonths):
    #for t in range(0,nmonths):
        # Calculate the corresponding months
        t_past=[q for q in range(t-t_immunity,t-1) if q>0]
        #kappa_I=[I[tt]*kappa[t-tt] for tt in t_past]
        #print t,t_past,[t-tt-1 for tt in t_past]
        sum_kappa_I=sum([I[tt]*kappa[t-tt-1] for tt in t_past])
        S[t]=(1.-frac_vacc)*(N-sum_kappa_I)
        #S[t]=N-sum_kappa_I
        R[t]=N-S[t]
        I[t]=beta_t[t]*I[t-1]**alpha*(S[t]/N)**gamma_*eps_t[t]
        #print t,sum_kappa_I

    #==============================================================
    #  Use pyplot to create a plot with dual y axes -- code modified from http://ask.sagemath.org/question/1000/two-y-axes
    #
    fig2 = plt.figure()
    # Create the first Y-axis on the left by default and
    # plot the first curve associated to the left Y-axis
    ax3 = fig2.add_subplot(111)
    ln3 = ax3.plot(years,I[0:-1], '-', color='red',label='Infected, $I(t)$')
    ax3.set_ylabel('# Individuals', color='black')
    #ax1.set_ylabel(ylabel='\# Individuals', color='black')
    #ax1.set_ylabel(ylabel='\# Individuals', color='red')

    # Paint the tick labels on the left Y-axis in red 
    # to match the color of a curve
    for tl in ax3.get_yticklabels():
        tl.set_color('red')

    # Create the second Y-axis on the right and
    # plot the second curve associated to the right Y-axis
    ax4 = ax3.twinx()
    ln4 = ax4.plot(years,S, '-', color='blue',label='Susceptible, $S(t)$')
    ax4.set_ylim([0,N])
    #ax2.set_ylabel(ylabel='\# Individuals', color='blue')

    # Paint the tick labels on the left Y-axis in blue 
    # to match the color of a curve
    for tl in ax4.get_yticklabels():
        tl.set_color('blue')

    # Create the vertical and horizontal grid lines
    ax3.xaxis.grid(color='grey', linestyle='--', linewidth=0.5)
    ax3.yaxis.grid(color='grey', linestyle='--', linewidth=0.5)
    #ax3.legend(handles=[ln3,ln4])
    ax3.legend(loc=(0.0625,0.75))
    ax4.legend(loc=(0.65,0.75))
    #ax4.legend(loc=2)
    
    # Set range in time
    plt.xlim([0.,nyears])
    plt.xlabel('Year')
    #if show_legend is True:
     #   # Create a combined legend
     #   lns=ln1+ln2
     #   labs=[l.get_label() for l in lns]
     #   ax3.legend(lns, labs, loc=0)

    # Save the figure (to see it in Sage)
    #plt.savefig(plot_prefix+'_comb.png')
    #fig.show()

    #show(I_plot+S_plot+R_plot)
    fig3 = plt.figure()
    ax5 = fig3.add_subplot(111)
    ax5.plot(kappa,'-',label=r'Immunity after $i$ mos., $\kappa_{i}$')
    ax5.set_xlabel(r'Months')
    ax5.set_ylabel(r'$\kappa_{i}$')
        #show(imm_plot,gridlines=True)
    # Some textual output
    print('Total number of cholera cases =',sum(I))
    print('Average monthly cholera cases =',sum(I)/len(I)) #mean(I))
    print('Average fraction of population that is susceptible =',sum(S)/(len(S)*N)) #mean(S)/N)
    #print('Average transmission rate =',mean([beta_t))
    print('Average transmission rate =',np.mean([beta_t[t]*eps_t[t] for t in range(1,nmonths)]))

In [ ]:
Nint=widgets.FloatText(value=200000,width=10,description = r"$N$")
alpha=widgets.FloatText(value=0.57,description = r"$\alpha$")
gamma=widgets.FloatText(value=1.,description = r"$\gamma$")
epsilon=widgets.FloatText(value=0.0,description = r"$\epsilon$")
Iinit_frac=widgets.FloatText(value=0.0001,description = r"$I_{init}$")
asymptom=widgets.FloatText(value=25.,description = r"$A$")
t_immunity=widgets.IntText(value=12,description = r"$t_{immunity}$")
frac_vacc=widgets.FloatText(value=0.,description = r"$f_{vaccinated}$")
ui2_1 = widgets.VBox([Nint,alpha,epsilon,Iinit_frac])
ui2_2 = widgets.VBox([asymptom,gamma,t_immunity,frac_vacc])

h = '30px'
L_Nint = widgets.Label(value='Initial Population',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="97%",height=h))
L_alpha = widgets.Label(value='Infected clustering',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="95%",height=h))
L_gamma = widgets.Label(value='Susceptible clustering',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="90%",height=h))
L_epsilon = widgets.Label(value='Noise',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="97%",height=h))
L_Iinit_frac = widgets.Label(value='Initial infected fraction',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-end",width="97%",height=h))
L_asymptom = widgets.Label(value='Asymptomatic ratio',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-start",width="97%",height=h))
L_t_immunity = widgets.Label(value='Immune period (months)',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-start",width="98%",height=h))
L_frac_vacc = widgets.Label(value='Fraction vaccinated',layout=widgets.Layout(display="flex", 
                                        justify_content="flex-start",width="95%",height=h))
ui2_L1 = widgets.VBox([L_Nint,L_alpha,L_epsilon,L_Iinit_frac])
ui2_L2 = widgets.VBox([L_asymptom,L_gamma,L_t_immunity,L_frac_vacc])
ui2_3 = widgets.HBox([ui2_L1,ui2_1,ui2_2,ui2_L2])

def plotCases(Nint,alpha,gamma,epsilon,Iinit_frac,asymptom,t_immunity,frac_vacc):
    plot_caseload(Nint=Nint,alpha=alpha,gamma_=gamma,epsilon = epsilon,
                  Iinit_frac=Iinit_frac,asymptom=asymptom,
                  t_immunity=t_immunity,frac_vacc=frac_vacc)
    
out2 = widgets.interactive_output(plotCases,{'Nint':Nint,'alpha':alpha,'gamma':gamma,
                                            'epsilon':epsilon,'Iinit_frac':Iinit_frac,
                                             'asymptom':asymptom,'t_immunity':t_immunity,
                                            'frac_vacc':frac_vacc})
display(ui2_3,out2)

:::{figure} #kc_2 
:placeholder: images/kc2.png
:align: left
:::